# EXP-009: Robust Per-Well Polynomial Projection Postprocess

**🖥 RUNS ON: Kaggle CPU — no GPU needed**

| Item | Value |
|------|-------|
| Target env | Kaggle CPU (no accelerator) |
| Compute | ~15 min |
| Expected gain | ~0.3–0.5 RMSE (top-score notebook, CV-validated) |
| Risk | Low — pure postprocess, no retraining |

**Run this BEFORE EXP-007** — cheapest experiment, applies to any prior prediction.

## What this does
Reproduced from `rogii-ridge-sp45-proj.ipynb` (LB 7.748):
- `U = TVT + Z - anchor`  where `anchor = last_known_TVT + last_known_Z`
- `s = (MD - MD_last) / (MD_end - MD_last)`  — normalized along-lateral position ∈ [0,1]
- Robust degree-5 polynomial fit via Tukey iterative reweighting (4 iters, c=2.0)
- `corrected_TVT = U_fit - Z + anchor`

In [ ]:
import numpy as np
import pandas as pd
import json
import time
from pathlib import Path
from sklearn.metrics import mean_squared_error

# ── Config ────────────────────────────────────────────────────────
EXP_ID      = 'EXP-009'
TINY_SAMPLE = False   # True = first 10 wells (still on Kaggle)
PROJ_DEG    = 5
PROJ_ITERS  = 4
PROJ_C      = 2.0

# ── Paths (Kaggle only) ───────────────────────────────────────────
INPUT_DIR = Path('/kaggle/input/rogii')
WORK_DIR  = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f'EXP_ID      = {EXP_ID}')
print(f'TINY_SAMPLE = {TINY_SAMPLE}')
print(f'Projection: deg={PROJ_DEG}, iters={PROJ_ITERS}, c={PROJ_C}')

In [ ]:
# ── Projection postprocess (inline — no external import needed) ───
def _robfit(s, u, deg=5, n_iter=4, c=2.0):
    """Robust polynomial fit via Tukey iterative reweighting."""
    w = np.ones(len(s))
    u_fit = u.copy()
    for _ in range(n_iter):
        coeffs = np.polyfit(s, u, deg=deg, w=w)
        u_fit  = np.polyval(coeffs, s)
        resid  = u - u_fit
        mad    = np.median(np.abs(resid - np.median(resid)))
        scale  = 1.4826 * mad if mad > 1e-12 else 1.0
        w      = (np.abs(resid) / scale <= c).astype(float)
        w      = np.maximum(w, 1e-6)
    return u_fit

def apply_projection(df, pred_col='tvt_pred', well_col='well_code',
                     md_col='MD', z_col='Z', tvt_col='TVT',
                     is_eval_col='is_eval', deg=5, n_iter=4, c=2.0):
    """Apply per-well robust projection to all eval rows in df."""
    corrected = df[pred_col].copy().astype(float)
    n_done = 0
    for well_id, wdf in df.groupby(well_col):
        wdf   = wdf.sort_values(md_col)
        known = wdf[~wdf[is_eval_col].astype(bool)]
        eval_ = wdf[wdf[is_eval_col].astype(bool)]
        if len(known) == 0 or len(eval_) < deg + 1:
            continue
        last   = known.iloc[-1]
        anchor = float(last[tvt_col]) + float(last[z_col])
        span   = float(eval_[md_col].iloc[-1]) - float(last[md_col])
        if span <= 0:
            continue
        s     = (eval_[md_col].values - float(last[md_col])) / span
        u     = eval_[pred_col].values + eval_[z_col].values - anchor
        u_fit = _robfit(s, u, deg=deg, n_iter=n_iter, c=c)
        corrected.loc[eval_.index] = u_fit - eval_[z_col].values + anchor
        n_done += 1
    print(f'  Projection applied to {n_done} wells')
    return corrected

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

print('Functions defined.')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────
train = pd.read_csv(INPUT_DIR / 'train.csv')
test  = pd.read_csv(INPUT_DIR / 'test.csv')

if TINY_SAMPLE:
    wells = train['well_code'].unique()[:10]
    train = train[train['well_code'].isin(wells)].copy()
    test  = test[test['well_code'].isin(wells)].copy()

print(f'Train: {train.shape}  Test: {test.shape}')
print(f'Wells — train: {train["well_code"].nunique()}  test: {test["well_code"].nunique()}')

In [ ]:
# ── Load prior OOF/test predictions ───────────────────────────────
# From EXP-007 final blend or aw pipeline (OOF RMSE 10.4521, LB 9.964).
# Priority: EXP-007_final_oof.npy > aw-blend_oof.npy > placeholder

for prior_name in ['EXP-007_final', 'aw-blend']:
    oof_path  = WORK_DIR / f'{prior_name}_oof.npy'
    test_path = WORK_DIR / f'{prior_name}_test.npy'
    if oof_path.exists():
        oof_pred_before  = np.load(oof_path)
        test_pred_before = np.load(test_path)
        print(f'Loaded prior OOF from: {prior_name}  shape: {oof_pred_before.shape}')
        break
else:
    print('WARNING: No prior OOF found — using noisy TVT as placeholder')
    rng = np.random.default_rng(42)
    oof_pred_before  = train['TVT'].values + rng.normal(0, 10, len(train))
    test_pred_before = rng.normal(1500, 100, len(test))

train['tvt_pred'] = oof_pred_before
test['tvt_pred']  = test_pred_before

In [ ]:
# ── Baseline RMSE ─────────────────────────────────────────────────
eval_mask   = train['is_eval'].astype(bool)
rmse_before = rmse(train.loc[eval_mask, 'TVT'].values,
                   train.loc[eval_mask, 'tvt_pred'].values)
print(f'OOF RMSE BEFORE projection: {rmse_before:.5f}')
print(f'Reference (aw pipeline):    10.45212  (LB 9.964)')

In [ ]:
# ── Apply projection — train OOF ──────────────────────────────────
t0 = time.time()
corrected_train = apply_projection(
    train, pred_col='tvt_pred', well_col='well_code',
    md_col='MD', z_col='Z', tvt_col='TVT', is_eval_col='is_eval',
    deg=PROJ_DEG, n_iter=PROJ_ITERS, c=PROJ_C
)
print(f'  Train done in {time.time()-t0:.1f}s')

# ── Apply projection — test ───────────────────────────────────────
# Build combined df: train known rows as anchor + test rows as eval
train_known = train[~eval_mask].copy()
train_known['is_eval'] = False
test_combined = pd.concat([
    train_known[['well_code', 'MD', 'Z', 'TVT', 'tvt_pred', 'is_eval']],
    test[['well_code', 'MD', 'Z', 'tvt_pred']].assign(TVT=np.nan, is_eval=True)
], ignore_index=True)

corrected_test_all = apply_projection(
    test_combined, pred_col='tvt_pred', well_col='well_code',
    md_col='MD', z_col='Z', tvt_col='TVT', is_eval_col='is_eval',
    deg=PROJ_DEG, n_iter=PROJ_ITERS, c=PROJ_C
)
test_pred_after = corrected_test_all[test_combined['is_eval'].astype(bool)].values
print(f'  Total: {time.time()-t0:.1f}s')

In [ ]:
# ── Evaluate ──────────────────────────────────────────────────────
rmse_after = rmse(train.loc[eval_mask, 'TVT'].values, corrected_train[eval_mask].values)

print('=' * 50)
print(f'OOF RMSE BEFORE: {rmse_before:.5f}')
print(f'OOF RMSE AFTER:  {rmse_after:.5f}')
print(f'Delta:           {rmse_after - rmse_before:+.5f}')
print('=' * 50)

wb, wa = [], []
for w, wdf in train[eval_mask].groupby('well_code'):
    idx = wdf.index
    wb.append(rmse(train.loc[idx, 'TVT'].values, train.loc[idx, 'tvt_pred'].values))
    wa.append(rmse(train.loc[idx, 'TVT'].values, corrected_train.loc[idx].values))
print(f'Per-well median RMSE — before: {np.median(wb):.4f}  after: {np.median(wa):.4f}')
print(f'% wells improved: {100*np.mean(np.array(wa) < np.array(wb)):.1f}%')

In [ ]:
# ── Save ──────────────────────────────────────────────────────────
np.save(WORK_DIR / f'{EXP_ID}_oof.npy',  corrected_train[eval_mask].values)
np.save(WORK_DIR / f'{EXP_ID}_test.npy', test_pred_after)

result = {
    'exp_id': EXP_ID,
    'rmse_before': round(rmse_before, 6), 'rmse_after': round(rmse_after, 6),
    'delta': round(rmse_after - rmse_before, 6),
    'pct_wells_improved': round(float(np.mean(np.array(wa) < np.array(wb))), 4),
    'proj_deg': PROJ_DEG, 'proj_iters': PROJ_ITERS, 'proj_c': PROJ_C,
    'tiny_sample': TINY_SAMPLE,
}
with open(WORK_DIR / f'{EXP_ID}_result.json', 'w') as f:
    json.dump(result, f, indent=2)
print(json.dumps(result, indent=2))
print('\nNext: record in experiment_queue.md + memory/previous_runs.md')

In [ ]:
# ── Generate submission ───────────────────────────────────────────
sample_sub = pd.read_csv(INPUT_DIR / 'sample_submission.csv')
sub = sample_sub.copy()
test['tvt_corrected'] = test_pred_after
sub = sub.merge(test[['row_id', 'tvt_corrected']], on='row_id', how='left')
sub['TVT'] = sub['tvt_corrected'].fillna(sub['TVT'])
sub = sub.drop(columns=['tvt_corrected'])
sub.to_csv(WORK_DIR / f'{EXP_ID}_submission.csv', index=False)
print(f'Submission saved: {len(sub)} rows')
sub.head()